# PrecisionMiner Workflow

This notebook demonstrates a complete extraction workflow on controlled paper sections. Use it to understand the workflow contract before running over PDFs.


## Setup And Evidence Index

The workflow retrieves evidence from a vector store and sends that evidence to a generator that returns JSON.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys
import tempfile


def find_project_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "episcope").exists():
            return candidate
    return None


PROJECT_ROOT = find_project_root(Path.cwd())
if PROJECT_ROOT is not None:
    src_path = str(PROJECT_ROOT / "src")
    if src_path not in sys.path:
        sys.path.insert(0, src_path)

WORK_DIR = Path(tempfile.mkdtemp(prefix="episcope-notebook-"))
WORK_DIR


In [ ]:
from episcope.schemas import PaperMetadata, StructuredSection

sample_papers = {
    "paper_open_data": {
        "metadata": PaperMetadata(
            title="Trial data sharing and reuse",
            abstract="A randomized trial reused a public patient-level dataset from a hospital registry.",
            keywords=["trial", "registry", "data sharing"],
        ),
        "sections": [
            StructuredSection(
                title="Methods",
                section_type="Methods",
                content=(
                    "The analysis used patient records from the National Hospital Registry. "
                    "The registry stores admission dates, treatment groups, and mortality outcomes."
                ),
            ),
            StructuredSection(
                title="Data availability",
                section_type="Data availability",
                content=(
                    "De-identified trial data and the analysis code are available from the public repository. "
                    "The dataset can be reused for non-commercial research after registration."
                ),
            ),
        ],
    },
    "paper_closed_data": {
        "metadata": PaperMetadata(
            title="Hospital cohort study",
            abstract="A cohort study collected clinical data directly from participating hospitals.",
            keywords=["cohort", "hospital", "mortality"],
        ),
        "sections": [
            StructuredSection(
                title="Participants",
                section_type="Methods",
                content=(
                    "The cohort included adult patients admitted to three hospitals. "
                    "Data were collected by the study team from electronic health records."
                ),
            ),
            StructuredSection(
                title="Data sharing",
                section_type="Data availability",
                content=(
                    "The patient dataset cannot be shared publicly because the consent agreement "
                    "does not permit redistribution of individual-level hospital records."
                ),
            ),
        ],
    },
}

list(sample_papers)


In [ ]:
import re
from typing import Iterable

import numpy as np

from episcope.rag.embeddings.base import Embedder


class TinyKeywordEmbedder(Embedder):
    vocabulary = (
        "data",
        "dataset",
        "source",
        "cohort",
        "survey",
        "registry",
        "trial",
        "patient",
        "hospital",
        "mortality",
        "covid",
        "treatment",
        "remdesivir",
        "supplement",
        "table",
        "figure",
        "reference",
        "database",
    )

    @property
    def model_name(self) -> str:
        return "tiny-keyword-demo"

    @property
    def dim(self) -> int:
        return len(self.vocabulary)

    def embed_text(self, text: str) -> list[float]:
        text = text.lower()
        counts = []
        for term in self.vocabulary:
            pattern = rf"\b{re.escape(term)}s?\b"
            counts.append(float(len(re.findall(pattern, text))))

        vector = np.array(counts, dtype="float32")
        norm = float(np.linalg.norm(vector))
        if norm:
            vector = vector / norm
        return vector.tolist()

    def embed_texts(self, texts: Iterable[str]) -> list[list[float]]:
        return [self.embed_text(text) for text in texts]


embedder = TinyKeywordEmbedder()
embedder.model_name, embedder.dim


In [ ]:
from episcope.rag.indexing.chunking import FixedSizeChunker
from episcope.rag.indexing.indexer import Indexer
from episcope.rag.retrieval.candidates import SemanticCandidateRetriever
from episcope.rag.retrieval.retriever import Retriever
from episcope.vectordb.file import FileDB

vdb = FileDB(str(WORK_DIR / "index"))
indexer = Indexer(
    vdb,
    embedder=embedder,
    chunker=FixedSizeChunker(chunk_size=500, chunk_overlap=50),
)

for paper_id, paper in sample_papers.items():
    indexer.index_paper(paper["sections"], paper["metadata"], paper_id=paper_id)

vdb.save()
semantic_candidates = SemanticCandidateRetriever(vdb, dense_embedder=embedder)
retriever = Retriever(vdb, candidate_retrievers=[semantic_candidates], use_rerank=False)

len(vdb.get_points()), vdb.get_embedding_model()


## Configure The Extraction Task

`FindDataSourcesConfig` provides retrieval prompts and the JSON schema used by the response parser.


In [ ]:
from episcope.db.in_memory_academic_db import InMemoryAcademicDB
from episcope.workflows.precision_miner import FindDataSourcesConfig

strategy_name = "demo-sections"
academic_db = InMemoryAcademicDB(backup_file=None)
for paper_id, paper in sample_papers.items():
    academic_db.insert(paper_id, "sections", strategy_name, [section.to_dict() for section in paper["sections"]])
    academic_db.insert(paper_id, "metadata", strategy_name, paper["metadata"].to_dict())
    academic_db.insert(paper_id, "references", strategy_name, [])

config = FindDataSourcesConfig(top_k=3)
config.retrieval_templates[:2], config.top_k


## Provide A Schema-Compatible Generator

Replace this class with `LLMGenerator` when you want model-backed extraction. The important contract is that `generate` returns JSON matching `ExtractionResult`.


In [ ]:
import json
from typing import Any, Callable, Optional, Sequence

from episcope.rag.generation.base import Generator
from episcope.rag.provenance import Evidence, Provenance


class FixedExtractionGenerator(Generator):
    model_id = "fixed-extraction-generator"

    def generate(
        self,
        contexts: Sequence[Any],
        *,
        question: Optional[str] = None,
        message_builder: Optional[Callable[..., Any]] = None,
        **kwargs: Any,
    ) -> Provenance:
        contexts = list(contexts)
        raw_text = contexts[0].text if contexts else ""
        payload = {
            "description": "The paper uses a named registry and public repository evidence.",
            "items": [
                {
                    "name": "National Hospital Registry",
                    "url": None,
                    "explanation": "The registry is named as the source of patient records and outcomes.",
                    "raw_text": raw_text,
                }
            ],
        }
        evidences = [
            Evidence(
                paper_id=getattr(ctx, "paper_id", ""),
                snippet=getattr(ctx, "text", ""),
                section=getattr(ctx, "section_type", None),
                model_id=self.model_id,
                prompt_id="fixed-demo",
            )
            for ctx in contexts
        ]
        return Provenance(answer=json.dumps(payload), evidences=evidences)


## Run And Inspect The Workflow

`run` returns the parsed extraction result. `run_detailed` also exposes prompts, raw generator output, retrieved chunks, and provenance.


In [ ]:
from episcope.workflows import PrecisionMiner

miner = PrecisionMiner(
    retriever=retriever,
    generator=FixedExtractionGenerator(),
    strategy_name=strategy_name,
    config=config,
    academic_db=academic_db,
)

detailed = miner.run_detailed("paper_open_data")
detailed.result.model_dump()


In [ ]:
print("Retrieved chunks")
for chunk in detailed.relevant_chunks:
    print(f"- {chunk.paper_id} | {chunk.section_title} | {chunk.text[:160]}")

print("\nRaw generator response")
print(detailed.trace.raw_llm_response)
